# Weak imposition of Dirichlet conditions for the Poisson problem
https://jsdokken.com/dolfinx-tutorial/chapter1/nitsche.html <br>
Idea: add additional terms to the variational formulation to impose boundary conditions instead of modifying the matrix system using strong imposition (lifting, i.e. enforcing that v=0 on the boundary).


In [1]:
from dolfinx import fem, mesh, plot, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
import numpy
from mpi4py import MPI
from ufl import (
    Circumradius,
    FacetNormal,
    SpatialCoordinate,
    TrialFunction,
    TestFunction,
    div,
    dx,
    ds,
    grad,
    inner,
)


In [2]:
# Create mesh
N = 8
domain = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
V = fem.functionspace(domain, ("Lagrange", 1))

Problem: $-\Delta u = f ~ on ~ \Omega$, <br> 
$u = u_0 ~ on ~ \partial \Omega$. With $\Omega = [0,1]^2$. <br>
Where $f(x,y)=-6,~ u_0(x,y)=1+x^2+2y^2$. This has the exact solution $u_e(x,y)=1+x^2+2y^2$.


In [3]:
# Create exact solution
x = SpatialCoordinate(domain)
u_ex = 1 + x[0]**2 + 2*x[1]**2
# Use exact solution for Dirichlet boundary
uD = fem.Function(V)
uD.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
f = -div(grad(u_ex)) # use symbolic differentiation capabilities of ufl to get negative laplacian of the exact solution to use directly in variational formulation

As opposed to the FEM_Tutorial_Book, the variational form is now changed: We start with integration by parts: <br>
$\int_\Omega \nabla u \cdot \nabla v dx - \int_{\partial\Omega} \nabla u \cdot nv ds = \int_\Omega fv dx$ <br>
Now we don't set $v=0$ on the boundary, but we add the following two terms to the variational formulation: <br>
$ -\int_{\partial\Omega} \nabla v \cdot n(u-u_D) ds + \alpha/h \int_{\partial\Omega} (u-u_D)v ds$ <br>
The first term enforces symmetry to the bilinear form and the latter enforces coercivity. $u_D$ is the known Dirichlet condition and $h$ is the diameter of the circumscribed sphere of the mesh element. Creating the bilinear and linear form $a$ and $L$: <br>
$a(u,v) = \int_\Omega \nabla u \cdot \nabla v dx + \int_{\partial\Omega} - (n\cdot \nabla u)v-(n\cdot \nabla v)u + \alpha/h uv ds$, <br>
$L(v)=\int_\Omega fv dx + \int_{\partial\Omega} - (n\cdot\nabla v)u_D + \alpha/h u_D v ds$

In [4]:
u = TrialFunction(V)
v = TestFunction(V)
n = FacetNormal(domain)
h = 2 * Circumradius(domain)
alpha = fem.Constant(domain, default_scalar_type(10))
a = inner(grad(u), grad(v)) * dx - inner(n, grad(u)) * v * ds
a += -inner(n, grad(v)) * u * ds + alpha / h * inner(u, v) * ds
L = inner(f, v) * dx
L += -inner(n, grad(v)) * uD * ds + alpha / h * inner(uD, v) * ds

In [5]:
# Solve the problem
problem = LinearProblem(a, L, petsc_options_prefix="nitsche_poisson")
uh = problem.solve()

In [6]:
# Compute error
error_form = fem.form(inner(uh - uD, uh - uD) * dx)
error_local = fem.assemble_scalar(error_form)
errorL2 = numpy.sqrt(domain.comm.allreduce(error_local, op=MPI.SUM))
if domain.comm.rank == 0:
    print(rf"$L^2$-error: {errorL2:.2e}")

$L^2$-error: 1.59e-03


In [7]:
# Compute max error at degrees of freedom
error_max = domain.comm.allreduce(
    numpy.max(numpy.abs(uD.x.array - uh.x.array)), op=MPI.MAX
)
if domain.comm.rank == 0:
    print(f"Error_max : {error_max:.2e}")

Error_max : 5.41e-03


Weakly imposed boundary condition -> we no longer fulfill the equation to machine precision at the mesh vertices

In [12]:
# Plot
import pyvista
pyvista.set_jupyter_backend("html")
grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(V))
grid.point_data["u"] = uh.x.array.real
grid.set_active_scalars("u")
plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True, show_scalar_bar=True)
plotter.view_xy()
if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    figure = plotter.screenshot("nitsche.png")

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…